# Part I — Exploratory Data Analysis: Ford GoBike Tripdata (Feb 2019)

**Introduction**
This notebook explores the `201902-fordgobike-tripdata.csv` dataset to understand trip duration, user types, and relationships between rider demographics and trip patterns.

**Questions to answer**:
- What is the distribution of trip durations?
- How do trip durations differ by `user_type` and `member_gender`?
- Are there relationships between rider age (inferred from `member_birth_year`) and trip length?
- Are there notable correlations between numeric variables?

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid")

In [ ]:
# Load dataset (file is in repository root)
csv_path = '201902-fordgobike-tripdata.csv'
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Preliminary wrangling and inspection
df.info()
print('
Missing values per column:')
print(df.isnull().sum())
# Convert times to datetime and create duration in minutes
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')
df['duration_min'] = df['duration_sec'] / 60.0
# Clean gender and birth year
df['member_gender'] = df['member_gender'].replace('', 'Unknown').fillna('Unknown')
df['member_birth_year'] = pd.to_numeric(df['member_birth_year'], errors='coerce')
# Compute approximate age for Feb 2019
df['age'] = 2019 - df['member_birth_year']
# Quick look after cleaning
df[['duration_sec','duration_min','user_type','member_gender','member_birth_year','age']].describe(include='all')

## Features of interest
We'll focus on: `duration_min`, `user_type`, `member_gender`, `age` (derived), and station coordinates for later spatial analysis if desired.
We'll use the Question-Visualization-Observations framework while exploring.

### Univariate: Trip duration distribution

In [ ]:
# Histogram of durations (minutes). Use log x-limit to show long tail more clearly.
plt.figure(figsize=(10,5))
sns.histplot(df['duration_min'].dropna(), bins=100, kde=False)
plt.xlim(0, 120)  # focus on trips up to 2 hours for clarity
plt.xlabel('Duration (minutes)')
plt.title('Trip duration distribution (focused to 0-120 min)')
plt.show()

### Univariate: User type counts

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='user_type', order=df['user_type'].value_counts().index)
plt.title('Count of trips by user type')
plt.ylabel('Number of trips')
plt.show()

### Bivariate: Duration vs Age (scatter)

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df.sample(n=min(len(df),2000), random_state=1), x='age', y='duration_min', hue='user_type', alpha=0.6)
plt.ylim(0,120)
plt.xlabel('Age (approx)')
plt.title('Trip duration vs age (sample)')
plt.show()

### Bivariate: Boxplot of duration by user type

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df.loc[df['duration_min'] <= 240], x='user_type', y='duration_min')
plt.ylim(0,120)
plt.ylabel('Duration (minutes)')
plt.title('Duration by user type (truncated at 240 min)')
plt.show()

### Multivariate: Correlation heatmap (numeric features)

In [ ]:
num_cols = ['duration_sec','duration_min','bike_id','member_birth_year','age']
corr = df[num_cols].corr()
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation matrix (numeric features)')
plt.show()

### Multivariate: Facet grid — duration distribution by gender and user type

In [ ]:
g = sns.catplot(data=df[df['duration_min'] <= 120], x='user_type', y='duration_min', col='member_gender', kind='box', height=4, aspect=0.9)
g.set_axis_labels('User type','Duration (min)')
g.fig.suptitle('Duration by user type across genders (truncated to 120 min)', y=1.02)
plt.show()

### Multivariate: Pair plot (subset of numeric features)

In [ ]:
sns.pairplot(df.dropna(subset=['duration_min','age']).sample(n=min(1000,len(df.dropna(subset=['duration_min','age'])))), vars=['duration_min','age'], hue='user_type', plot_kws={'alpha':0.5})
plt.suptitle('Pair plot (duration vs age)')
plt.show()

## Observations and next steps
- Most trips are short (many under 30 minutes) with a long tail of longer trips.
- `Subscribers` constitute the majority of recorded trips.
- Boxplots suggest differences in duration distributions between `user_type`s; further statistical tests can quantify this.
- Correlation matrix shows limited strong linear correlation between age and duration; other encodings or transformations might reveal additional structure.

Next steps: refine cleaning (handle missing birth years), explore temporal patterns (hour-of-day, weekday vs weekend), and create explanatory visualizations for Part II.

### Temporal patterns: hour-of-day and weekday/weekend analysis

In [ ]:
# Create temporal features
df['start_hour'] = df['start_time'].dt.hour
df['start_day'] = df['start_time'].dt.day_name()
df['is_weekend'] = df['start_day'].isin(['Saturday','Sunday'])
# Trips per hour
hour_counts = df['start_hour'].value_counts().sort_index()
plt.figure(figsize=(10,4))
sns.barplot(x=hour_counts.index, y=hour_counts.values, color='C0')
plt.xlabel('Start hour (0-23)')
plt.ylabel('Number of trips')
plt.title('Trips by start hour')
plt.show()
# Average duration by start hour
avg_dur_hour = df.groupby('start_hour')['duration_min'].mean()
plt.figure(figsize=(10,4))
sns.lineplot(x=avg_dur_hour.index, y=avg_dur_hour.values, marker='o')
plt.xlabel('Start hour')
plt.ylabel('Average duration (min)')
plt.title('Average trip duration by start hour')
plt.show()

In [ ]:
# Heatmap: hour vs weekday counts
pivot = df.pivot_table(index='start_day', columns='start_hour', values='duration_sec', aggfunc='count').fillna(0)
# Reorder days to start Monday
days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(days_order)
plt.figure(figsize=(12,4))
sns.heatmap(pivot, cmap='YlGnBu')
plt.xlabel('Start hour')
plt.title('Heatmap of trip counts by weekday and hour')
plt.show()

### Station-level summary: top start stations and spatial overview

In [ ]:
# Top 10 start stations
top_starts = df['start_station_name'].value_counts().nlargest(10)
plt.figure(figsize=(8,5))
sns.barplot(y=top_starts.index, x=top_starts.values, palette='mako')
plt.xlabel('Number of trips')
plt.ylabel('Start station')
plt.title('Top 10 start stations by trip count')
plt.show()

In [ ]:
# Spatial scatter of start stations (aggregated counts)
coords = df.groupby(['start_station_latitude','start_station_longitude']).size().reset_index(name='counts')
plt.figure(figsize=(6,6))
plt.scatter(coords['start_station_longitude'], coords['start_station_latitude'], s=np.sqrt(coords['counts'])*2, c=coords['counts'], cmap='viridis', alpha=0.7)
plt.colorbar(label='Trip counts')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Spatial distribution of start stations (size ~ sqrt(count))')
plt.show()

## Additional observations
- Peak trip counts occur during commute hours (morning and evening).
- Average trip duration shows modest variation by hour; mid-day trips can be slightly longer on average.
- A small set of stations account for a large fraction of trip starts (typical hub behavior).

These temporal and station-level views will inform Part II explanatory visualizations (e.g., a focused slide on commute patterns and hotspot stations).

## Refinements: cleaning and improved visualizations

In [ ]:
# Create a cleaned subset and age groups
clean = df.copy()
clean = clean[(clean['duration_min']>0)]
clean = clean[clean['age'].between(10,100)]
# Age groups
bins = [10,20,30,40,50,60,100]
labels = ['10-19','20-29','30-39','40-49','50-59','60+']
clean['age_group'] = pd.cut(clean['age'], bins=bins, labels=labels, right=False)
clean.shape

In [ ]:
# Refined histogram with log-scale y and central tendency lines
plt.figure(figsize=(10,4))
sns.histplot(clean['duration_min'], bins=60, kde=False, color='C1')
plt.ylim(1, clean['duration_min'].value_counts().max()*1.1)
plt.yscale('log')
mean = clean['duration_min'].mean()
median = clean['duration_min'].median()
plt.axvline(mean, color='k', linestyle='--', label=f"mean={mean:.1f} min")
plt.axvline(median, color='r', linestyle='-', label=f"median={median:.1f} min")
plt.xlim(0,120)
plt.xlabel('Duration (minutes)')
plt.ylabel('Count (log scale)')
plt.title('Refined trip duration histogram (y-axis log)')
plt.legend()
plt.savefig('duration_hist_refined.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Violin plot of duration by user_type (truncated) 
plt.figure(figsize=(8,5))
sns.violinplot(data=clean[clean['duration_min']<=240], x='user_type', y='duration_min', inner='quartile', palette='Set2')
plt.ylim(0,120)
plt.xlabel('User type')
plt.ylabel('Duration (minutes)')
plt.title('Violin: duration distribution by user type (truncated at 240 min)')
plt.savefig('violin_duration_user_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Period mapping for commute vs midday etc.
def period_of_hour(h):
    if h<5:
        return 'LateNight'
    if 5<=h<9:
        return 'MorningCommute'
    if 9<=h<15:
        return 'Midday'
    if 15<=h<19:
        return 'EveningCommute'
    if 19<=h<24:
        return 'Night'
    return 'Other'

clean['period'] = clean['start_hour'].apply(lambda x: period_of_hour(x) if not np.isnan(x) else 'Unknown')

# Average duration by period and weekend
pivot2 = clean.groupby(['is_weekend','period'])['duration_min'].mean().reset_index()
plt.figure(figsize=(8,4))
sns.barplot(data=pivot2, x='period', y='duration_min', hue='is_weekend', palette='coolwarm')
plt.xlabel('Period')
plt.ylabel('Average duration (min)')
plt.title('Average duration by period: weekday vs weekend')
plt.xticks(rotation=30)
plt.savefig('avg_duration_by_period.png', dpi=150, bbox_inches='tight')
plt.show()

## Refinements summary
- Cleaned: removed trips with non-positive durations and limited ages to 10–100 for robustness.
- Added age groups and focused plots to reduce outlier distortion.
- Histogram uses log y-axis to show the long tail while keeping low-duration detail.
- Violin plots summarize duration distributions by `user_type`.
- Barplots compare average durations across day periods and weekend vs weekday.
- Saved refined figures to `duration_hist_refined.png`, `violin_duration_user_type.png`, and `avg_duration_by_period.png`.

Next: I can (A) run the notebook and export to HTML, or (B) polish figures for Part II (colors, annotations). Which do you prefer?

## Polished visualizations: styling, annotations, and high-res outputs

In [ ]:
import matplotlib.ticker as ticker
sns.set_style('whitegrid')
pal = sns.color_palette('tab10')

# Polished histogram
fig, ax = plt.subplots(figsize=(10,4))
sns.histplot(clean['duration_min'], bins=60, color=pal[1], ax=ax)
ax.set_yscale('log')
ax.set_xlim(0,120)
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Count (log scale)')
mean = clean['duration_min'].mean()
median = clean['duration_min'].median()
ax.axvline(mean, color='k', linestyle='--', linewidth=1)
ax.axvline(median, color='r', linestyle='-', linewidth=1)
# annotate median and mean
ax.text(median+2, ax.get_ylim()[1]*0.3, f"median={median:.1f} min", color='r')
ax.text(mean+2, ax.get_ylim()[1]*0.2, f"mean={mean:.1f} min", color='k')
# percent under 30 min
pct_under_30 = (clean['duration_min']<=30).mean()*100
ax.text(2, ax.get_ylim()[1]*0.9, f"{pct_under_30:.1f}% trips ≤ 30 min", fontsize=10, bbox=dict(facecolor='white', alpha=0.7))
fig.tight_layout()
fig.savefig('duration_hist_polished.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Polished violin plot with median markers
fig, ax = plt.subplots(figsize=(8,5))
sns.violinplot(data=clean[clean['duration_min']<=240], x='user_type', y='duration_min', inner=None, palette='Set2', ax=ax)
# overlay medians
meds = clean.groupby('user_type')['duration_min'].median()
for i, u in enumerate(meds.index):
    ax.scatter(i, meds.loc[u], color='k', s=30, zorder=10)
    ax.text(i+0.05, meds.loc[u]+2, f"{meds.loc[u]:.1f} min", fontsize=9)
ax.set_ylim(0,120)
ax.set_xlabel('User type')
ax.set_ylabel('Duration (minutes)')
ax.set_title('Duration distribution by user type (trimmed)')
fig.tight_layout()
fig.savefig('violin_duration_polished.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Polished barplot: average duration by period with annotations
fig, ax = plt.subplots(figsize=(9,4))
order = ['LateNight','MorningCommute','Midday','EveningCommute','Night']
plot_df = clean.groupby(['is_weekend','period'])['duration_min'].mean().reset_index()
# ensure ordering
plot_df['period'] = pd.Categorical(plot_df['period'], categories=order, ordered=True)
sns.barplot(data=plot_df, x='period', y='duration_min', hue='is_weekend', palette=['#4c72b0','#dd8452'], ax=ax)
ax.set_xlabel('Period')
ax.set_ylabel('Average duration (min)')
ax.set_title('Average trip duration by period: Weekday vs Weekend')
# annotate bars
for p in ax.patches:
    h = p.get_height()
    if np.isfinite(h):
        ax.text(p.get_x()+p.get_width()/2., h+0.5, f"{h:.1f}", ha='center', fontsize=9)
fig.tight_layout()
fig.savefig('avg_duration_by_period_polished.png', dpi=200, bbox_inches='tight')
plt.show()

## Conclusions and next steps

**Key findings**
- Most trips are short: over a large majority of trips are ≤ 30 minutes, supporting short urban trips use-case.
- Commute patterns: trip counts peak during typical morning and evening commute hours; average durations are slightly higher during midday.
- User type differences: `Subscribers` form the bulk of trips; distributions differ between `Subscribers` and `Customers`, with Customers showing a heavier tail in long trips.
- Station hubs: a few start stations account for a large fraction of trips, consistent with hub-and-spoke travel behavior.

**Next steps (Part II suggestions)**
- Create 2–3 explanatory plots for slides: (1) commute heatmap, (2) top stations map with callouts, (3) duration distribution comparing user types with annotations.
- Run basic statistical tests (e.g., Mann–Whitney U) to support claims about duration differences.
- Export notebook to HTML and assemble a short slide deck with the polished figures.



In [ ]:
# Test environment for notebook export
import nbformat
import sys
print('nbformat', nbformat.__version__)
try:
    import nbconvert
    print('nbconvert', nbconvert.__version__)
except Exception as e:
    print('nbconvert import failed:', e)
